# 🗺️ POI RAG System - モジュール使用例

このNotebookでは、`src/`ディレクトリの共通モジュールを使用してRAGシステムを構築・評価する方法を示します。

**前提条件**:
- Google Driveにsrcフォルダがアップロードされていること
- GPUランタイムが有効になっていること

---
## Step 1: 環境セットアップ

In [ ]:
%%capture
# パッケージインストール
!pip install -q transformers accelerate bitsandbytes
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers
!pip install -q tqdm pandas matplotlib japanize-matplotlib requests

print("✅ パッケージインストール完了")

In [ ]:
# Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

print("✅ Google Driveマウント完了")

In [ ]:
# srcモジュールをPythonパスに追加
import sys
sys.path.insert(0, '/content/drive/MyDrive/experiments-local-llm')

# モジュールインポート
from src import (
    TestCase, TEST_CASES,
    count_keyword_hits, has_coordinate, has_poi_name, calculate_score, TestResult,
    setup_directories, save_results, generate_report
)
from src.utils import fetch_poi_from_osm, detect_category
from src.evaluators import aggregate_results, aggregate_by_category

print("✅ srcモジュールインポート完了")
print(f"   テストケース数: {len(TEST_CASES)}件")

---
## Step 2: ディレクトリ準備

In [ ]:
# ディレクトリ設定
dirs = setup_directories()

print("✅ ディレクトリ準備完了")
for key, path in dirs.items():
    print(f"   {key}: {path}")

---
## Step 3: POIデータ取得

In [ ]:
import json
import os

# POIデータのパス
poi_path = f"{dirs['data']}/poi_documents.json"

# 既存データがあれば読み込み、なければ取得
if os.path.exists(poi_path):
    print("既存のPOIデータを読み込み中...")
    with open(poi_path, "r", encoding="utf-8") as f:
        poi_documents = json.load(f)
    print(f"✅ POIデータ読み込み完了: {len(poi_documents)}件")
else:
    print("Overpass APIからPOIデータを取得中...")
    area_config = {
        "name": "渋谷駅周辺",
        "bbox": "35.655,139.695,35.665,139.710"
    }
    poi_documents = fetch_poi_from_osm(area_config)
    
    # 保存
    with open(poi_path, "w", encoding="utf-8") as f:
        json.dump(poi_documents, f, ensure_ascii=False, indent=2)
    print(f"✅ POIデータ取得・保存完了: {len(poi_documents)}件")

---
## Step 4: モデルロード

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from langchain_huggingface import HuggingFaceEmbeddings

# モデル設定
LLM_MODEL = "Qwen/Qwen2.5-7B-Instruct"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

# 量子化設定
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# LLMロード
print(f"Loading {LLM_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)
print(f"✅ LLMロード完了")

# Embeddingロード
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': 'cuda'},
    encode_kwargs={'normalize_embeddings': True}
)
print(f"✅ Embeddingロード完了")

---
## Step 5: RAGシステム構築

In [ ]:
from src.rag_system import create_rag_system

# RAGシステム構築
rag_system = create_rag_system(
    model=model,
    tokenizer=tokenizer,
    poi_documents=poi_documents,
    embeddings=embeddings,
    debug=True
)

print("✅ RAGシステム構築完了")

---
## Step 6: テスト実行

In [ ]:
from tqdm import tqdm
import time

def run_single_test(rag_system, test_case: TestCase, verbose: bool = True) -> TestResult:
    """単一テストを実行"""
    if verbose:
        print(f"\n[Test {test_case.id}] {test_case.prompt[:40]}...")
    
    # RAGあり
    rag_result = rag_system.query_with_rag(test_case.prompt)
    
    # RAGなし
    no_rag_result = rag_system.query_without_rag(test_case.prompt)
    
    # 評価
    rag_kw_hits = count_keyword_hits(rag_result["answer"], test_case.expected_keywords)
    no_rag_kw_hits = count_keyword_hits(no_rag_result["answer"], test_case.expected_keywords)
    
    rag_has_coord = has_coordinate(rag_result["answer"])
    no_rag_has_coord = has_coordinate(no_rag_result["answer"])
    
    rag_has_name = has_poi_name(rag_result["answer"], poi_documents)
    no_rag_has_name = has_poi_name(no_rag_result["answer"], poi_documents)
    
    # スコア計算
    rag_score = calculate_score(
        rag_kw_hits, len(test_case.expected_keywords),
        rag_has_coord, rag_has_name, test_case.expected_data_type
    )
    no_rag_score = calculate_score(
        no_rag_kw_hits, len(test_case.expected_keywords),
        no_rag_has_coord, no_rag_has_name, test_case.expected_data_type
    )
    
    if verbose:
        print(f"  RAG: {rag_score:.1f} / NoRAG: {no_rag_score:.1f} / 改善: {rag_score - no_rag_score:+.1f}")
    
    return TestResult(
        test_id=test_case.id,
        test_category=test_case.category,
        prompt=test_case.prompt,
        rag_answer=rag_result["answer"],
        rag_time_ms=rag_result["time_ms"],
        rag_keyword_hits=rag_kw_hits,
        rag_keyword_total=len(test_case.expected_keywords),
        rag_has_coordinate=rag_has_coord,
        rag_has_poi_name=rag_has_name,
        no_rag_answer=no_rag_result["answer"],
        no_rag_time_ms=no_rag_result["time_ms"],
        no_rag_keyword_hits=no_rag_kw_hits,
        no_rag_has_coordinate=no_rag_has_coord,
        no_rag_has_poi_name=no_rag_has_name,
        rag_score=rag_score,
        no_rag_score=no_rag_score,
        improvement=rag_score - no_rag_score,
        difficulty=test_case.difficulty
    )

# クイックテスト（5件）
print("=" * 60)
print("クイックテスト（5件）")
print("=" * 60)

quick_results = []
for tc in TEST_CASES[:5]:
    result = run_single_test(rag_system, tc, verbose=True)
    quick_results.append(result)

# サマリー
summary = aggregate_results(quick_results)
print(f"\n【クイックテスト結果】")
print(f"  平均RAGスコア: {summary['avg_rag_score']:.1f}")
print(f"  平均NoRAGスコア: {summary['avg_no_rag_score']:.1f}")
print(f"  平均改善率: {summary['avg_improvement']:+.1f}")

---
## Step 7: 結果保存

In [ ]:
# 結果保存
result_path = save_results(
    results=quick_results,
    output_dir=dirs['results'],
    prefix="quick_test",
    metadata={
        "llm_model": LLM_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "poi_count": len(poi_documents)
    }
)

print(f"✅ 結果保存: {result_path}")

In [ ]:
# レポート生成
report = generate_report(
    results_summary=summary,
    model_info={
        "llm_model": LLM_MODEL,
        "embedding_model": EMBEDDING_MODEL,
        "poi_count": len(poi_documents)
    },
    raspi_baseline={
        "avg_rag_score": 68.1,
        "avg_rag_time_ms": 318000
    },
    output_path=f"{dirs['results']}/quick_test_report.md"
)

print(report)

---
## 🎉 完了

共通モジュールを使用したテスト実行が完了しました。

**作成されたファイル**:
- `results/quick_test_YYYYMMDD_HHMMSS.json`: テスト結果
- `results/quick_test_report.md`: レポート